In [ ]:
#!/usr/bin/env python3
"""
rfsoc_server.py — TCP server for the RFSoC 4x2 (v4).

Protocol: Each transaction begins with a 1-byte command ID.

  CMD 0x01 — Transfer:
    Client → Server:  [1B cmd=0x01] [4B uint32 num_samples_full]
                       [num_samples_full * 2B int16 I]
                       [num_samples_full * 2B int16 Q]
    Server → Client:  [4B uint32 num_out]
                       [num_out * 8B float64 I_rec]
                       [num_out * 8B float64 Q_rec]

  CMD 0x02 — Set DAC Current (VOP):
    Client → Server:  [1B cmd=0x02] [4B uint32 vop_uA]
    Server → Client:  [1B status (1=ok, 0=err)]

  CMD 0x03 — Set ADC Attenuation (DSA):
    Client → Server:  [1B cmd=0x03] [8B float64 att_dB]
    Server → Client:  [1B status (1=ok, 0=err)]

  CMD 0x04 — Set DAC Nyquist Zone:
    Client → Server:  [1B cmd=0x04] [1B uint8 zone (1 or 2)]
    Server → Client:  [1B status (1=ok, 0=err)]

  CMD 0x05 — Set DAC NCO Frequency:
    Client → Server:  [1B cmd=0x05] [8B float64 freq_mhz]
    Server → Client:  [1B status (1=ok, 0=err)]

  CMD 0x06 — Set ADC Nyquist Zone:
    Client → Server:  [1B cmd=0x06] [1B uint8 zone (1 or 2)]
    Server → Client:  [1B status (1=ok, 0=err)]

  CMD 0x07 — Set ADC NCO Frequency:
    Client → Server:  [1B cmd=0x07] [8B float64 freq_mhz]
    Server → Client:  [1B status (1=ok, 0=err)]
"""

import sys
sys.path.append('/home/xilinx/jupyter_notebooks/MyLibs/')

import socket
import struct
import time
import numpy as np
from pynq import Overlay, allocate
import xrfdc
import traceback

# ============================================================
# System parameters
# ============================================================
M_ADC = 4
M_DAC = 4
NUM_SAMPLES_MAX = 200_000  # Maximum samples per channel (safety limit)

HOST = '0.0.0.0'
PORT = 5555

# Safe defaults
DEFAULT_DAC_VOP = 10000   # µA — niedrig, sicher
DEFAULT_ADC_ATT = 11.0    # dB — maximum attenuation

# ============================================================
# FPGA setup
# ============================================================
print("[INIT] Loading bitstream ...")
overlay = Overlay("experiment.bit")
hls_core = overlay.mm2s_replay_engine_V2_0
dma_recv = overlay.axi_dma_0.recvchannel

# Initialize TLAST controller
tlast_controller = overlay.tlast_controller

# RFDC for DAC/ADC control
rfdc = overlay.usp_rf_data_converter_0
dac_tile = rfdc.dac_tiles[0]
dac_block = dac_tile.blocks[0]
adc_tile = rfdc.adc_tiles[0]
adc_block = adc_tile.blocks[0]

# Set safe initial values
dac_block.SetDACVOP(DEFAULT_DAC_VOP)
adc_block.DSA = {'Attenuation': DEFAULT_ADC_ATT}
print(f"[INIT] DAC VOP = {DEFAULT_DAC_VOP} µA, ADC DSA = {DEFAULT_ADC_ATT} dB")
print("[INIT] Bitstream loaded.")


def _recv_exact(sock, nbytes):
    chunks = []
    received = 0
    while received < nbytes:
        chunk = sock.recv(min(nbytes - received, 65536))
        if not chunk:
            raise ConnectionError("Connection closed")
        chunks.append(chunk)
        received += len(chunk)
    return b''.join(chunks)


# =============================================================
# PACKING/UNPACKING
# =============================================================
def pack_and_load(I, Q, input_buffer):
    I_pairs = I.reshape(-1, 2)
    Q_pairs = Q.reshape(-1, 2)
    I0 = (I_pairs[:, 0] & 0xFFFF).astype(np.uint64)
    I1 = (I_pairs[:, 1] & 0xFFFF).astype(np.uint64)
    Q0 = (Q_pairs[:, 0] & 0xFFFF).astype(np.uint64)
    Q1 = (Q_pairs[:, 1] & 0xFFFF).astype(np.uint64)
    packed_data = (Q1 << 48) | (I1 << 32) | (Q0 << 16) | I0
    hls_core.register_map.enable = 0
    
    # Fill the allocated buffer completely
    input_buffer[:] = packed_data
    input_buffer.flush()


def unpack_output(output_buffer, num_samples):
    output_buffer.invalidate()
    raw_out = output_buffer.view(np.uint64)
    
    I_rec = np.zeros(num_samples * M_ADC)
    Q_rec = np.zeros(num_samples * M_ADC)
    raw_I = raw_out[0::2]
    raw_Q = raw_out[1::2]
    
    I_rec[0::4] = (raw_I & 0xFFFF).astype(np.int16)
    I_rec[1::4] = ((raw_I >> 16) & 0xFFFF).astype(np.int16)
    I_rec[2::4] = ((raw_I >> 32) & 0xFFFF).astype(np.int16)
    I_rec[3::4] = ((raw_I >> 48) & 0xFFFF).astype(np.int16)
    Q_rec[0::4] = (raw_Q & 0xFFFF).astype(np.int16)
    Q_rec[1::4] = ((raw_Q >> 16) & 0xFFFF).astype(np.int16)
    Q_rec[2::4] = ((raw_Q >> 32) & 0xFFFF).astype(np.int16)
    Q_rec[3::4] = ((raw_Q >> 48) & 0xFFFF).astype(np.int16)
    return I_rec, Q_rec


# =============================================================
# PIPELINE
# =============================================================
def run_fpga_pipeline(I, Q, num_samples):
    # Dynamically allocate buffers sized to the incoming signal
    input_buffer = allocate(shape=(num_samples * (M_DAC >> 1),), dtype=np.uint64)
    output_buffer = allocate(shape=(num_samples * 2,), dtype=np.uint64)

    try:
        pack_and_load(I, Q, input_buffer)
        
        # Set TLAST dynamically based on signal length (value, mask)
        tlast_controller.channel1.write(num_samples - 1, 0xFFFF_FFFF)
        
        hls_core.register_map.ddr_memory_1 = input_buffer.device_address
        hls_core.register_map.size = num_samples
        hls_core.write(0x00, 0x81)
        hls_core.register_map.enable = 1
        time.sleep(0.1)
        
        dma_recv.transfer(output_buffer)
        dma_recv.wait()
        
        return unpack_output(output_buffer, num_samples)
    
    finally:
        # Release PYNQ DMA buffers
        input_buffer.close()
        output_buffer.close()


# =============================================================
# RFDC CONTROL
# =============================================================
def set_dac_vop(vop_uA):
    """Set DAC output current (µA). Valid range: ~6425–32000."""
    dac_block.SetDACVOP(int(vop_uA))
    print(f"[DAC]  VOP = {vop_uA} µA")

def set_adc_att(att_dB):
    """Set ADC step attenuator (dB). Range: 0.0–11.0 in 0.5 dB steps."""
    att_dB = round(att_dB * 2) / 2  # round to nearest 0.5 dB step
    att_dB = max(0.0, min(11.0, att_dB))
    adc_block.DSA = {'Attenuation': att_dB}
    print(f"[ADC]  DSA = {att_dB} dB (readback: {adc_block.DSA})")

def set_dac_nyquist(zone):
    if zone not in [1, 2]:
        raise ValueError("Nyquist Zone must be 1 or 2.")
    dac_block.NyquistZone = zone
    print(f"[DAC]  Nyquist Zone = {zone}")

def set_dac_nco(freq_mhz):
    mixer_settings = dac_block.MixerSettings
    mixer_settings['Freq'] = float(freq_mhz)
    dac_block.MixerSettings = mixer_settings
    dac_block.UpdateEvent(xrfdc.EVENT_MIXER)
    print(f"[DAC]  NCO Frequency = {freq_mhz} MHz")

def set_adc_nyquist(zone):
    if zone not in [1, 2]:
        raise ValueError("Nyquist Zone must be 1 or 2.")
    adc_block.NyquistZone = zone
    print(f"[ADC]  Nyquist Zone = {zone}")

def set_adc_nco(freq_mhz):
    mixer_settings = adc_block.MixerSettings
    mixer_settings['Freq'] = float(freq_mhz)
    adc_block.MixerSettings = mixer_settings
    adc_block.UpdateEvent(xrfdc.EVENT_MIXER)
    print(f"[ADC]  NCO Frequency = {freq_mhz} MHz")


# =============================================================
# TCP SERVER — Command-Dispatcher
# =============================================================
CMD_TRANSFER    = 0x01
CMD_DAC_VOP     = 0x02
CMD_ADC_ATT     = 0x03
CMD_DAC_NYQUIST = 0x04
CMD_DAC_NCO     = 0x05
CMD_ADC_NYQUIST = 0x06
CMD_ADC_NCO     = 0x07

def handle_client(conn, addr):
    print(f"[CONN] Client connected: {addr}")
    try:
        while True:
            # Read 1-byte command ID
            cmd_byte = conn.recv(1)
            if not cmd_byte:
                break
            cmd = cmd_byte[0]

            if cmd == CMD_TRANSFER:
                header = _recv_exact(conn, 4)
                num_samples_full = struct.unpack('<I', header)[0]
                num_samples = num_samples_full // M_DAC 

                if num_samples > NUM_SAMPLES_MAX:
                    print(f"[ERR]  Received samples ({num_samples}) exceed maximum ({NUM_SAMPLES_MAX}). Aborting.")
                    nbytes_iq = num_samples_full * 2
                    _recv_exact(conn, nbytes_iq) 
                    _recv_exact(conn, nbytes_iq) 
                    conn.sendall(struct.pack('<I', 0))
                    continue

                nbytes_iq = num_samples_full * 2
                I_in = np.frombuffer(_recv_exact(conn, nbytes_iq), dtype=np.int16).copy()
                Q_in = np.frombuffer(_recv_exact(conn, nbytes_iq), dtype=np.int16).copy()

                print(f"[RUN]  Transferring {num_samples} samples per channel ...")
                I_rec, Q_rec = run_fpga_pipeline(I_in, Q_in, num_samples)

                num_out = len(I_rec)
                conn.sendall(struct.pack('<I', num_out))
                conn.sendall(I_rec.tobytes())
                conn.sendall(Q_rec.tobytes())
                print(f"[DONE] {num_out} samples sent.")

            elif cmd == CMD_DAC_VOP:
                data = _recv_exact(conn, 4)
                vop = struct.unpack('<I', data)[0]
                try:
                    set_dac_vop(vop)
                    conn.sendall(bytes([1]))
                except Exception as e:
                    print(f"[ERR]  DAC VOP: {e}")
                    conn.sendall(bytes([0]))

            elif cmd == CMD_ADC_ATT:
                data = _recv_exact(conn, 8)
                att = struct.unpack('<d', data)[0]
                try:
                    set_adc_att(att)
                    conn.sendall(bytes([1]))
                except Exception as e:
                    print(f"[ERR]  ADC ATT: {e}")
                    conn.sendall(bytes([0]))

            elif cmd == CMD_DAC_NYQUIST:
                data = _recv_exact(conn, 1)
                zone = struct.unpack('<B', data)[0]  # <B for uint8
                try:
                    set_dac_nyquist(zone)
                    conn.sendall(bytes([1]))
                except Exception as e:
                    print(f"[ERR]  DAC Nyquist: {e}")
                    conn.sendall(bytes([0]))

            elif cmd == CMD_DAC_NCO:
                data = _recv_exact(conn, 8)
                freq = struct.unpack('<d', data)[0]  # <d for float64
                try:
                    set_dac_nco(freq)
                    conn.sendall(bytes([1]))
                except Exception as e:
                    print(f"[ERR]  DAC NCO: {e}")
                    conn.sendall(bytes([0]))

            elif cmd == CMD_ADC_NYQUIST:
                data = _recv_exact(conn, 1)
                zone = struct.unpack('<B', data)[0]
                try:
                    set_adc_nyquist(zone)
                    conn.sendall(bytes([1]))
                except Exception as e:
                    print(f"[ERR]  ADC Nyquist: {e}")
                    conn.sendall(bytes([0]))

            elif cmd == CMD_ADC_NCO:
                data = _recv_exact(conn, 8)
                freq = struct.unpack('<d', data)[0]
                try:
                    set_adc_nco(freq)
                    conn.sendall(bytes([1]))
                except Exception as e:
                    print(f"[ERR]  ADC NCO: {e}")
                    conn.sendall(bytes([0]))

            else:
                print(f"[ERR]  Unknown command: 0x{cmd:02X}")

    except ConnectionError:
        print(f"[DISC] {addr} disconnected.")
    except Exception:
        traceback.print_exc()
    finally:
        conn.close()


def main():
    srv = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    srv.bind((HOST, PORT))
    srv.listen(1)
    print(f"[SRV]  Listening on {HOST}:{PORT}")

    while True:
        conn, addr = srv.accept()
        handle_client(conn, addr)


if __name__ == '__main__':
    main()